# Affirmations — LangChain + LangGraph + Ollama

This notebook demonstrates generating structured affirmations for spiritual practices using:

- **Ollama** (local Gemma 3 4B model) — fully local inference, no API keys for the LLM
- **LangGraph agent** — research-augmented generation with tool calling
- **Research Tools**: SearxNG (self-hosted metasearch)
- **Trafilatura** — best-in-class HTML content extraction for the research processing layer
- **Pydantic v2** — structured output validation

## Prerequisites

1. Run `nx run affirmations:setup` to start Ollama + SearxNG and pull `gemma3:4b`
2. Ollama available at `http://localhost:11434`
3. SearxNG available at `http://localhost:8889`

## Architecture

```
User Query
    │
    ▼
LangGraph ReAct Agent
    │
    └─► searxng_search ──► research.py (Trafilatura) ──► Context Budget
    │
    ▼
Structured Affirmation (Pydantic)
```


## Model


In [4]:
from langchain_ollama import ChatOllama


model = "gemma3:4b"
temperature = 0.48
base_url = "http://ollama:11434"
llm = ChatOllama(base_url=base_url, model=model, temperature=temperature)
print(f'LLM: model {llm.model} at {llm.base_url} with temperature={llm.temperature}')


LLM: model gemma3:4b at http://ollama:11434 with temperature=0.48


## Test Model


In [ ]:
hello_world = llm.invoke("Hello, world!")
print(f'Hello World Response: {hello_world}')

## Tools


In [ ]:
from langchain_community.utilities import SearxSearchWrapper
from langchain_core.tools import BaseTool, tool
from langchain_core.messages import HumanMessage, ToolMessage

searxng_wrapper = SearxSearchWrapper(searx_host="http://searxng:8080")

@tool
def searxng_search(query: str) -> str:
    """Search using SearxNG, a self-hosted metasearch engine aggregating
    results from various sources. Use for comprehensive research on the web.

    Args:
        query: The search terms to look up, phrased as a concise natural language query.
    """
    results = searxng_wrapper.run(query)
    extracted = trafilatura.extract(results, include_comments=False, include_tables=True, no_fallback=False)
    return f"[Source: searxng] {query}: {extracted}"

tools: list[BaseTool] = [searxng_search]

print(f'Available tools ({len(tools)} total):')
for tool in tools:
    print(f'  • {tool.name}: {tool.description}')

## Test Tools


In [ ]:
llm_with_tools = llm.bind_tools(tools)

human_message = HumanMessage(content="What is the meaning of the tarot card "The Hermit"?")
llm_response = llm_with_tools.invoke([human_message])

print("=== LLM response (tool call) ===")
print(f"content   : {llm_response.content!r}")
print(f"tool_calls: {llm_response.tool_calls}")

tool_call = llm_response.tool_calls[0]
tool_result = searxng_search.invoke(tool_call)

tool_message = ToolMessage(
    content=tool_result,
    tool_call_id=tool_call["id"],
)

print("\n=== Tool result ===")
print(tool_message.content[:300])

# Step 3 — Feed result back; model produces a final answer
final = llm_with_tools.invoke([human_message, llm_response, tool_message])

print("\n=== Final LLM response ===")
print(final.content)

## Agent


In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

RESEARCH_AGENT_SYSTEM_PROMPT = """You are an expert researcher and affirmation writer specializing in spiritual practices.

Your workflow for generating affirmations:
1. RESEARCH: Use the available tools to gather information about the spiritual practice and topic.
   - Use searxng_search for comprehensive results across multiple sources including Wikipedia

2. GENERATE: Based on your research, create a rich, meaningful affirmation that incorporates
   the authentic symbolism, history, and spiritual significance of the topic.

Research results are pre-processed and condensed — trust the summaries rather than
requesting redundant searches on the same topic.

When generating affirmations:
- Follow the requested grammatical structure exactly
- Draw on authentic symbolism and meanings from your research
- Make the affirmation personal (use "I"), uplifting, and spiritually resonant
- Include relevant keywords from the spiritual practice
"""

agent = create_agent(
    model=llm,
    tools=tools,
    prompt=SystemMessage(content=RESEARCH_AGENT_SYSTEM_PROMPT),
)

print('Research agent created.')
print(f'Graph nodes: {list(agent.get_graph().nodes.keys())}')


## Run Agent


In [ ]:
from langchain_core.messages import HumanMessage

# Invoke the agent for a Tower tarot affirmation
# The agent will research first, then generate
user_request = (
    'Research The Tower tarot card (XVI) — its symbolism, history, and meaning. '
    'Then generate an affirmation for someone working with The Tower energy, '
    'using the structure: "I am [positive quality] through [transformative process]"'
)

print('Invoking research agent...')
print(f'Request: {user_request[:100]}...')
print()

result = agent.invoke({'messages': [HumanMessage(content=user_request)]})

# Display the agent's reasoning trace
print('=== Agent Reasoning Trace ===')
for i, msg in enumerate(result['messages']):
    msg_type = type(msg).__name__
    content_preview = str(msg.content)[:200] if msg.content else '(tool call)'
    print(f'[{i}] {msg_type}: {content_preview}')
    print()

print('=== Final Agent Response ===')
print(result['messages'][-1].content)